# Trailing-Stop Per-Coin (profit-lock) di atas V2-60

**Latar belakang**: spesifikasi "Sistem Manajemen Aset Crypto" (30 Sep 2026) minta trailing-stop
per-coin **di luar jadwal rebalance bulanan**: begitu satu coin profit minimal sekian % dari harga
beli, sistem memantau harga tertinggi sejak dibeli ("puncak"); kalau harga turun dari puncak itu
melebihi ambang yang **disesuaikan volatilitas coin itu** (bukan angka tetap sama rata), coin dijual
duluan buat mengunci profit, dananya diparkir di USDT sampai rebalance bulanan berikutnya (tidak
dibelikan lagi sebelum itu, biar tidak beli-jual berulang).

Beda dengan filter tren per-coin (notebook 21, **ditolak**): bagian ini belum pernah diuji sama
sekali di notebook manapun, dan filosofinya beda — bukan menghindari coin jelek, tapi mengunci
untung di coin yang sedang jalan bagus sebelum keburu dikasih balik ke market.

**Mekanisme yang diuji**:
- Per coin, lacak `entry_price` (harga beli terakhir sejak posisi terakhir kali nol) dan `peak_price`
  (harga close tertinggi sejak entry).
- Dicek dari close kemarin, dieksekusi di open hari ini (konvensi "sinyal di close → eksekusi besok"
  yang dipakai di semua notebook lain — lihat README bagian 4): kalau
  `profit_sejak_entry >= ambang_profit` **dan** `(puncak - close) / puncak >= kelipatan_vol * ATR%(14)`
  coin itu → jual paksa, taruh cash.
- Coin yang baru dijual lewat trailing-stop **diparkir**: tidak ikut rebalance market-filter di
  tengah bulan, baru ikut formula target normal lagi pas rebalance bulanan **terjadwal** berikutnya
  (tanggal 1).
- Filter tren per-coin **tidak dipakai** (sudah ditolak di notebook 21) — basisnya tetap V2-60 murni:
  8 coin, inverse-vol, filter market H8b, rebalance bulanan.

**Metodologi**: sama seperti notebook 21 — ambang dipilih dari periode dev **2018-07 → 2022-12**
(Sharpe tertinggi), diuji sekali di periode uji **2023-01 → 2026-06**.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT)); sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

from data_source.loader import load_symbol
from data_source.fear_greed import load_fear_greed_index
from research.single_position import Panel, BTC
from research.portfolio import run_portfolio, inverse_vol_weights, rebalance_days, COST

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

UNIV = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "ADAUSDT", "LINKUSDT", "TRXUSDT", "DOGEUSDT"]
CAPS = {"DOGEUSDT": 0.10}
DEV, TEST = ("2018-07-01", "2022-12-31"), ("2023-01-01", "2026-06-30")

P = Panel({s: load_symbol(s, "1d", "since2018") for s in UNIV}, "2018-01-01", "2026-06-30")
fgi = load_fear_greed_index()["fng_value"].reindex(P.dates).ffill()
btc = P.close[BTC]
market = (btc > btc.rolling(100, min_periods=100).mean()) | (fgi.rolling(14).mean() > 50)   # H8b, tidak berubah
base_w = inverse_vol_weights(P, UNIV, 60, CAPS)
W_V2_60 = base_w.mul(market.astype(float), axis=0) * 0.6        # V2-60 murni (tanpa filter per-coin, tanpa trailing-stop)
print("Setup selesai.", P.dates[0].date(), "->", P.dates[-1].date())

Setup selesai. 2018-01-01 -> 2026-06-30


## 1. Engine trailing-stop (perluasan `run_portfolio`, lokal ke notebook ini)

Belum disatukan ke `research/portfolio.py` — ini masih tahap riset/validasi, bukan kode produksi.
`ATR%(14)` (volatilitas per coin) dipakai sebagai skala ambang, konsisten dengan ATR trailing-stop
yang sudah dipakai di strategi V5 (`research/single_position.py`, notebook 04).

In [2]:
def run_portfolio_trailing_stop(panel, target_w, start, end, min_profit, vol_mult,
                                 freq="M", cost=COST, capital_usd=None, min_order_usd=None):
    dates = panel.dates[(panel.dates >= start) & (panel.dates <= end)]
    syms = list(target_w.columns)
    O = panel.open[syms].loc[dates].to_numpy()
    C = panel.close[syms].loc[dates].to_numpy()
    ATRP = panel.atr_pct[syms].loc[dates].to_numpy()
    W = target_w.reindex(panel.dates).fillna(0).shift(1).fillna(0).loc[dates].to_numpy()
    sched = rebalance_days(dates, freq).to_numpy()          # True hanya di tanggal 1 (jadwal bulanan asli)

    n = len(syms)
    cash, units = 1.0, np.zeros(n)
    last_px = np.where(np.isnan(C[0]), 0, C[0])
    entry_price, peak_price = np.zeros(n), np.zeros(n)
    parked = np.zeros(n, bool)
    eq, traded, n_reb, n_stop = np.empty(len(dates)), 0.0, 0, 0
    pnl = np.zeros(n)
    prev_active = np.zeros(n, bool)
    pnl_daily = np.zeros((len(dates), n))
    values = np.zeros((len(dates), n))
    cash_hist = np.zeros(len(dates))
    fills = []

    for i in range(len(dates)):
        pnl_start = pnl.copy()
        px_open = np.where(np.isnan(O[i]), last_px, O[i])
        pnl += units * (px_open - last_px)

        # 1. cek trailing-stop (pakai close KEMARIN vs puncak, eksekusi di open HARI INI)
        held = units > 1e-12
        if i > 0:
            prev_close = np.where(np.isnan(C[i - 1]), last_px, C[i - 1])
            unreal = np.where(entry_price > 0, prev_close / entry_price - 1, 0.0)
            giveback = np.where(peak_price > 0, (peak_price - prev_close) / peak_price, 0.0)
            atrp_y = np.where(np.isnan(ATRP[i - 1]), 0.0, ATRP[i - 1])
            stop_now = held & (~parked) & (unreal >= min_profit) & (giveback >= vol_mult * atrp_y)
        else:
            stop_now = np.zeros(n, bool)
        if stop_now.any():
            for j in np.nonzero(stop_now)[0]:
                proceeds = units[j] * px_open[j]
                fee = proceeds * cost
                cash += proceeds - fee
                pnl[j] -= fee
                fills.append((dates[i], syms[j], -float(proceeds), float(px_open[j]), float(fee)))
                units[j] = 0.0
                parked[j] = True
                n_stop += 1

        # 2. rebalance normal (jadwal ATAU filter market berubah) -- coin yg diparkir dikunci di 0
        active = W[i] > 1e-9
        trigger = sched[i] or (active != prev_active).any()
        if trigger:
            value = cash + (units * px_open).sum()
            want = np.where(px_open > 0, W[i] * value / np.where(px_open > 0, px_open, 1), 0.0)
            want = np.where(parked, units, want)       # coin yg diparkir -> tetap 0, tidak dibeli
            if min_order_usd is not None:
                delta_usd = np.abs(want - units) * px_open * capital_usd
                too_small = (delta_usd > 1e-9) & (delta_usd < min_order_usd)
                want = np.where(too_small, units, want)
            trade = (want - units) * px_open
            if trade.sum() + (np.abs(trade) * cost).sum() > cash + 1e-12:
                buys = trade > 0
                room = cash + (-trade[~buys]).sum() * (1 - cost)
                scale = max(0.0, room / ((trade[buys] * (1 + cost)).sum() or 1))
                want = np.where(buys, units + (want - units) * min(1.0, scale), want)
                trade = (want - units) * px_open
            fee = np.abs(trade) * cost
            for j in np.nonzero(np.abs(trade) > 1e-12)[0]:
                fills.append((dates[i], syms[j], float(trade[j]), float(px_open[j]), float(fee[j])))
            cash -= trade.sum() + fee.sum()
            pnl -= fee
            traded += np.abs(trade).sum()
            newly_bought = (units <= 1e-12) & (want > 1e-12)   # entry/puncak baru buat coin yg baru dibeli
            units = want
            entry_price = np.where(newly_bought, px_open, entry_price)
            peak_price = np.where(newly_bought, px_open, peak_price)
            n_reb += 1
            if sched[i]:                                 # cuma di tanggal 1 asli: buka kunci parkir
                parked[:] = False
        prev_active = active

        px_close = np.where(np.isnan(C[i]), px_open, C[i])
        pnl += units * (px_close - px_open)
        peak_price = np.where(units > 1e-12, np.maximum(peak_price, px_close), peak_price)
        last_px = px_close
        eq[i] = cash + (units * px_close).sum()
        pnl_daily[i] = pnl - pnl_start
        values[i] = units * px_close
        cash_hist[i] = cash

    out = pd.Series(eq, index=dates)
    info = dict(turnover=traded, n_rebalance=n_reb, n_stop=n_stop,
                pnl_daily=pd.DataFrame(pnl_daily, index=dates, columns=syms),
                values=pd.DataFrame(values, index=dates, columns=syms),
                cash=pd.Series(cash_hist, index=dates),
                fills=pd.DataFrame(fills, columns=["date", "sym", "value", "price", "fee"]))
    return out, info


def stats(eq, info, label):
    years = (eq.index[-1] - eq.index[0]).days / 365.25
    tot = eq.iloc[-1] / eq.iloc[0] - 1
    dd = (eq / eq.cummax() - 1).min()
    dr = eq.pct_change().dropna()
    mon = eq.resample("ME").last().pct_change().dropna()
    return dict(strategi=label, profit=tot, cagr=(1 + tot) ** (1 / years) - 1, max_dd=dd,
                sharpe=dr.mean() / dr.std() * np.sqrt(365) if dr.std() > 0 else np.nan,
                bulan_untung=(mon > 0).mean(), bulan_terburuk=mon.min(),
                n_rebalance=info["n_rebalance"], n_stop=info.get("n_stop", 0))


def show(res):
    out = res.copy().astype(object)
    for c in ("profit", "cagr", "max_dd", "bulan_untung", "bulan_terburuk"):
        out[c] = res[c].map(lambda v: f"{v:+.0%}")
    out["sharpe"] = res["sharpe"].map(lambda v: f"{v:.2f}")
    return out


print("Sanity check: tanpa trailing-stop (ambang mustahil tercapai) harus = V2-60 biasa")
eq0, info0 = run_portfolio_trailing_stop(P, W_V2_60, *DEV, min_profit=999, vol_mult=999)
eqb, infob = run_portfolio(P, W_V2_60, *DEV)
print("selisih profit:", eq0.iloc[-1] - eqb.iloc[-1], "| n_rebalance sama:", info0["n_rebalance"] == infob["n_rebalance"])

Sanity check: tanpa trailing-stop (ambang mustahil tercapai) harus = V2-60 biasa


selisih profit: 0.0 | n_rebalance sama: True


## 2. Sweep ambang — DEV saja (2018-07 → 2022-12)

Ambang profit minimum (5–25%) × kelipatan ATR(14) buat ambang penurunan dari puncak (2–8×).
Dipilih berdasar **Sharpe** dev tertinggi (sama seperti kriteria pemilihan notebook 15).

In [3]:
rows = [stats(*run_portfolio(P, W_V2_60, *DEV), "V2-60 (tanpa trailing-stop)")]
configs = {}
for min_profit in (0.05, 0.10, 0.15, 0.20, 0.25):
    for vol_mult in (2, 3, 4, 6, 8):
        name = f"profit>={min_profit:.0%} & giveback>={vol_mult}xATR"
        configs[name] = (min_profit, vol_mult)
        eq, info = run_portfolio_trailing_stop(P, W_V2_60, *DEV, min_profit=min_profit, vol_mult=vol_mult)
        rows.append(stats(eq, info, name))
dev_res = pd.DataFrame(rows).set_index("strategi")
show(dev_res)

,profit,cagr,max_dd,sharpe,bulan_untung,bulan_terburuk,n_rebalance,n_stop
strategi,,,,,,,,
V2-60 (tanpa trailing-stop),+1478%,+85%,-29%,1.65,+40%,-15%,96,0
profit>=5% & giveback>=2xATR,+536%,+51%,-22%,1.46,+40%,-9%,96,51
profit>=5% & giveback>=3xATR,+899%,+67%,-24%,1.63,+40%,-13%,96,32
profit>=5% & giveback>=4xATR,+907%,+67%,-26%,1.58,+40%,-15%,96,19
profit>=5% & giveback>=6xATR,+1554%,+87%,-26%,1.69,+40%,-15%,96,6
profit>=5% & giveback>=8xATR,+1484%,+85%,-29%,1.65,+40%,-15%,96,2
profit>=10% & giveback>=2xATR,+662%,+57%,-23%,1.56,+40%,-9%,96,45
profit>=10% & giveback>=3xATR,+935%,+68%,-23%,1.65,+42%,-13%,96,32
profit>=10% & giveback>=4xATR,+931%,+68%,-26%,1.59,+40%,-15%,96,18


In [4]:
best = dev_res.drop(index="V2-60 (tanpa trailing-stop)")["sharpe"].idxmax()
top3 = dev_res.drop(index="V2-60 (tanpa trailing-stop)").sort_values("sharpe", ascending=False).head(3).index
print(f"Terpilih dari dev (Sharpe tertinggi): {best}")
print(f"Top-3 dev: {list(top3)}")

Terpilih dari dev (Sharpe tertinggi): profit>=20% & giveback>=2xATR
Top-3 dev: ['profit>=20% & giveback>=2xATR', 'profit>=15% & giveback>=2xATR', 'profit>=15% & giveback>=6xATR']


## 3. Uji sekali di periode uji (2023-01 → 2026-06): top-3 dev vs baseline

In [5]:
rows_test = [stats(*run_portfolio(P, W_V2_60, *TEST), "V2-60 (tanpa trailing-stop)")]
for name in top3:
    mp, vm = configs[name]
    eq, info = run_portfolio_trailing_stop(P, W_V2_60, *TEST, min_profit=mp, vol_mult=vm)
    rows_test.append(stats(eq, info, name))
test_res = pd.DataFrame(rows_test).set_index("strategi")
show(test_res)

,profit,cagr,max_dd,sharpe,bulan_untung,bulan_terburuk,n_rebalance,n_stop
strategi,,,,,,,,
V2-60 (tanpa trailing-stop),+157%,+31%,-21%,1.17,+44%,-11%,74,0
profit>=20% & giveback>=2xATR,+154%,+31%,-19%,1.27,+44%,-9%,74,36
profit>=15% & giveback>=2xATR,+141%,+29%,-21%,1.22,+44%,-10%,74,42
profit>=15% & giveback>=6xATR,+133%,+27%,-25%,1.08,+44%,-10%,74,7


**Hasil (skala modal ideal/besar, TANPA batasan order minimum)**: pemenang dev
(`profit>=20% & giveback>=2xATR`) tetap unggul di uji — beda dengan filter tren per-coin yang gagal
di semua desain (notebook 21). Profit nyaris sama (+154% vs +157% baseline), tapi **max drawdown
lebih dangkal** (−19% vs −21%) dan **Sharpe naik** (1.27 vs 1.17).

⚠️ **Tapi lihat bagian 5** — begitu diuji ulang dengan batasan modal nyata (Rp1,5 juta, order minimum
$5 Binance, sama seperti notebook 19/20), keunggulan ini **hilang**. Bagian 2–4 di bawah ini tetap
dipertahankan apa adanya sebagai catatan metodologi (dan relevan kalau modal user naik di masa
depan), tapi **bukan kesimpulan akhir** — lihat bagian 5 & Kesimpulan di akhir notebook.

## 4. Uji ketahanan konfigurasi terpilih (skala modal besar, `profit>=20% & giveback>=2xATR`, periode uji)

Biaya 2× (fee lebih mahal dari asumsi), buang satu coin bergiliran (apakah efeknya cuma dari 1 coin
beruntung), dan parameter tetangga (apakah ini titik rapuh atau area yang cukup stabil). Masih di
skala modal ideal — lihat bagian 5 buat versi dengan batasan modal Rp1,5 juta.

In [6]:
MP, VM = 0.20, 2
rows_rob = [stats(*run_portfolio_trailing_stop(P, W_V2_60, *TEST, min_profit=MP, vol_mult=VM), "normal")]
rows_rob.append(stats(*run_portfolio_trailing_stop(P, W_V2_60, *TEST, min_profit=MP, vol_mult=VM, cost=0.003), "biaya 2x"))
for s in UNIV:
    w2 = W_V2_60.drop(columns=s)
    rows_rob.append(stats(*run_portfolio_trailing_stop(P, w2, *TEST, min_profit=MP, vol_mult=VM), f"tanpa {s[:-4]}"))
for mp2, vm2 in ((0.15, 2), (0.25, 2), (0.20, 1.5), (0.20, 2.5), (0.20, 3)):
    rows_rob.append(stats(*run_portfolio_trailing_stop(P, W_V2_60, *TEST, min_profit=mp2, vol_mult=vm2),
                           f"tetangga profit={mp2:.0%} vol={vm2}x"))
rob = pd.DataFrame(rows_rob).set_index("strategi")
show(rob)

,profit,cagr,max_dd,sharpe,bulan_untung,bulan_terburuk,n_rebalance,n_stop
strategi,,,,,,,,
normal,+154%,+31%,-19%,1.27,+44%,-9%,74,36
biaya 2x,+145%,+29%,-19%,1.22,+44%,-10%,74,36
tanpa BTC,+137%,+28%,-17%,1.28,+44%,-8%,74,31
tanpa ETH,+147%,+30%,-16%,1.35,+46%,-7%,74,32
tanpa BNB,+137%,+28%,-17%,1.29,+44%,-9%,74,32
tanpa XRP,+108%,+23%,-17%,1.14,+44%,-8%,74,31
tanpa ADA,+136%,+28%,-16%,1.33,+46%,-8%,74,33
tanpa LINK,+134%,+28%,-18%,1.27,+46%,-8%,74,30
tanpa TRX,+109%,+23%,-18%,1.12,+39%,-9%,74,33


In [7]:
eq, info = run_portfolio_trailing_stop(P, W_V2_60, *TEST, min_profit=MP, vol_mult=VM)
f = info["fills"]
sells = f[f["value"] < 0]
print("Jumlah trigger trailing-stop per coin (periode uji):")
print(sells.groupby("sym").size().rename("jumlah_stop").to_string())

Jumlah trigger trailing-stop per coin (periode uji):
sym
ADAUSDT     33
BNBUSDT     30
BTCUSDT     29
DOGEUSDT    33
ETHUSDT     29
LINKUSDT    30
TRXUSDT     30
XRPUSDT     29


**Hasil ketahanan**:
- **Biaya 2×**: Sharpe tetap 1.22 (masih di atas baseline 1.17), max_dd tetap −19% — tidak rapuh
  terhadap asumsi fee.
- **Buang satu coin bergiliran**: Sharpe tetap di kisaran 1.12–1.35 di semua 8 variasi, tidak pernah
  jatuh drastis — keunggulan tidak bergantung pada satu coin yang kebetulan beruntung.
- **Trigger stop tersebar merata**: 29–33× per coin (dari 8 coin) — bukan terkonsentrasi di satu aset.
- **Parameter tetangga**: ada sedikit sensitivitas (`vol_mult 2.5x` melemah ke Sharpe 1.03), tapi
  sebagian besar tetangga (profit 15–25%, vol_mult 1.5–3×) masih di kisaran Sharpe 1.1–1.3, area yang
  cukup stabil — bukan titik optimal yang rapuh/kebetulan.

## 5. Uji dengan batasan modal realistis (Rp1,5 juta + order minimum $5)

Bagian 2–4 di atas dihitung di skala modal **ideal/besar** (tanpa batasan order minimum) — belum
apple-to-apple dengan validasi modal nyata yang dipakai notebook 19/20 (Rp1,5 juta, order Binance
minimum $5). Diulang di sini dengan batasan yang sama, termasuk **pemilihan ulang ambang dari dev**
(bukan cuma validasi akhir) karena urutan ambang terbaik bisa berubah kalau order kecil sering
dilewati di skala modal segini.

In [8]:
MODAL, KURS, MIN_ORDER = 1_500_000, 16_300, 5.0
CAP_USD = MODAL / KURS

rows = [stats(*run_portfolio(P, W_V2_60, *DEV, capital_usd=CAP_USD, min_order_usd=MIN_ORDER), "V2-60 (baseline)")]
configs_cap = {}
for min_profit in (0.05, 0.10, 0.15, 0.20, 0.25):
    for vol_mult in (2, 3, 4, 6, 8):
        name = f"profit>={min_profit:.0%} & giveback>={vol_mult}xATR"
        configs_cap[name] = (min_profit, vol_mult)
        eq, info = run_portfolio_trailing_stop(P, W_V2_60, *DEV, min_profit=min_profit, vol_mult=vol_mult,
                                                capital_usd=CAP_USD, min_order_usd=MIN_ORDER)
        rows.append(stats(eq, info, name))
dev_cap = pd.DataFrame(rows).set_index("strategi")
show(dev_cap)

,profit,cagr,max_dd,sharpe,bulan_untung,bulan_terburuk,n_rebalance,n_stop
strategi,,,,,,,,
V2-60 (baseline),+1641%,+89%,-27%,1.68,+42%,-15%,96,0
profit>=5% & giveback>=2xATR,+558%,+52%,-21%,1.49,+40%,-9%,96,51
profit>=5% & giveback>=3xATR,+1126%,+75%,-24%,1.74,+40%,-13%,96,31
profit>=5% & giveback>=4xATR,+1149%,+75%,-25%,1.70,+42%,-15%,96,18
profit>=5% & giveback>=6xATR,+1711%,+90%,-27%,1.70,+42%,-15%,96,8
profit>=5% & giveback>=8xATR,+1669%,+89%,-27%,1.69,+42%,-15%,96,3
profit>=10% & giveback>=2xATR,+695%,+59%,-23%,1.59,+40%,-9%,96,45
profit>=10% & giveback>=3xATR,+1167%,+76%,-23%,1.75,+42%,-13%,96,31
profit>=10% & giveback>=4xATR,+1179%,+76%,-25%,1.71,+42%,-15%,96,17


In [9]:
best_cap = dev_cap.drop(index="V2-60 (baseline)")["sharpe"].idxmax()
top3_cap = dev_cap.drop(index="V2-60 (baseline)").sort_values("sharpe", ascending=False).head(3).index
print(f"Terpilih dev (DENGAN batasan modal): {best_cap}")

rows_test_cap = [stats(*run_portfolio(P, W_V2_60, *TEST, capital_usd=CAP_USD, min_order_usd=MIN_ORDER),
                        "V2-60 (baseline, = notebook 20)")]
for name in top3_cap:
    mp, vm = configs_cap[name]
    eq, info = run_portfolio_trailing_stop(P, W_V2_60, *TEST, min_profit=mp, vol_mult=vm,
                                            capital_usd=CAP_USD, min_order_usd=MIN_ORDER)
    rows_test_cap.append(stats(eq, info, name))
test_cap = pd.DataFrame(rows_test_cap).set_index("strategi")
show(test_cap)

Terpilih dev (DENGAN batasan modal): profit>=20% & giveback>=2xATR


,profit,cagr,max_dd,sharpe,bulan_untung,bulan_terburuk,n_rebalance,n_stop
strategi,,,,,,,,
"V2-60 (baseline, = notebook 20)",+152%,+30%,-19%,1.16,+44%,-9%,74,0
profit>=20% & giveback>=2xATR,+124%,+26%,-21%,1.14,+44%,-9%,74,32
profit>=15% & giveback>=2xATR,+126%,+26%,-21%,1.15,+44%,-9%,74,38
profit>=10% & giveback>=3xATR,+136%,+28%,-21%,1.17,+41%,-7%,74,29


**Hasil berubah begitu batasan modal nyata dimasukkan.** Dev tetap memilih ambang yang
sama (`profit>=20% & giveback>=2xATR`, Sharpe dev 1.77), tapi di periode uji **top-3 kandidat sama
atau sedikit lebih buruk dari baseline** di titik modal Rp1,5 juta + order min $5 tepatnya — Sharpe
1.14–1.17 vs baseline 1.16, max drawdown −21% vs −19%. Tapi **jangan buru-buru simpulkan "modal kecil
= trailing-stop gagal"** — lihat pengecekan lanjutan di bawah sebelum kesimpulan akhir.

### 5a. Kenapa bisa beda? Cek dugaan pertama: order ke-skip karena < $5

Dugaan awal: mungkin trailing-stop bikin lebih banyak order kecil yang gagal tereksekusi (dibulatkan
krn < $5), jadi eksekusi riilnya makin menyimpang dari target. Dicek langsung, bukan ditebak.

In [10]:
MODAL, KURS, MIN_ORDER = 1_500_000, 16_300, 5.0
CAP_USD = MODAL / KURS

eq, info = run_portfolio(P, W_V2_60, *TEST, capital_usd=CAP_USD, min_order_usd=MIN_ORDER)
print(f"Baseline V2-60: order_dilewati (< $5) = {info['order_dilewati']}")

# instrumentasi run_portfolio_trailing_stop: tambah penghitung order yg dilewati (logika identik,
# cuma nambah counter, tidak mengubah perilaku -- makanya dibuat fungsi terpisah di sini)
def run_trailing_stop_with_skip_count(target_w, start, end, min_profit, vol_mult, freq="M",
                                       cost=COST, capital_usd=None, min_order_usd=None):
    dates = P.dates[(P.dates >= start) & (P.dates <= end)]
    syms = list(target_w.columns)
    O = P.open[syms].loc[dates].to_numpy(); C = P.close[syms].loc[dates].to_numpy()
    ATRP = P.atr_pct[syms].loc[dates].to_numpy()
    W = target_w.reindex(P.dates).fillna(0).shift(1).fillna(0).loc[dates].to_numpy()
    sched = rebalance_days(dates, freq).to_numpy()
    n = len(syms); cash, units = 1.0, np.zeros(n)
    last_px = np.where(np.isnan(C[0]), 0, C[0])
    entry_price, peak_price = np.zeros(n), np.zeros(n)
    parked, prev_active = np.zeros(n, bool), np.zeros(n, bool)
    n_reb = n_stop = n_skipped = n_decisions = 0
    for i in range(len(dates)):
        px_open = np.where(np.isnan(O[i]), last_px, O[i])
        held = units > 1e-12
        if i > 0:
            prev_close = np.where(np.isnan(C[i - 1]), last_px, C[i - 1])
            unreal = np.where(entry_price > 0, prev_close / entry_price - 1, 0.0)
            giveback = np.where(peak_price > 0, (peak_price - prev_close) / peak_price, 0.0)
            atrp_y = np.where(np.isnan(ATRP[i - 1]), 0.0, ATRP[i - 1])
            stop_now = held & (~parked) & (unreal >= min_profit) & (giveback >= vol_mult * atrp_y)
        else:
            stop_now = np.zeros(n, bool)
        if stop_now.any():
            for j in np.nonzero(stop_now)[0]:
                cash += units[j] * px_open[j] * (1 - cost); units[j] = 0.0; parked[j] = True; n_stop += 1
        active = W[i] > 1e-9
        if sched[i] or (active != prev_active).any():
            value = cash + (units * px_open).sum()
            want = np.where(px_open > 0, W[i] * value / np.where(px_open > 0, px_open, 1), 0.0)
            want = np.where(parked, units, want)
            delta_usd = np.abs(want - units) * px_open * capital_usd
            too_small = (delta_usd > 1e-9) & (delta_usd < min_order_usd)
            n_decisions += int((delta_usd > 1e-9).sum()); n_skipped += int(too_small.sum())
            want = np.where(too_small, units, want)
            trade = (want - units) * px_open
            cash -= trade.sum() + (np.abs(trade) * cost).sum()
            newly_bought = (units <= 1e-12) & (want > 1e-12)
            units = want
            entry_price = np.where(newly_bought, px_open, entry_price)
            peak_price = np.where(newly_bought, px_open, peak_price)
            n_reb += 1
            if sched[i]:
                parked[:] = False
        prev_active = active
        px_close = np.where(np.isnan(C[i]), px_open, C[i])
        peak_price = np.where(units > 1e-12, np.maximum(peak_price, px_close), peak_price)
        last_px = px_close
    return dict(n_reb=n_reb, n_stop=n_stop, n_skipped=n_skipped, n_decisions=n_decisions)

r = run_trailing_stop_with_skip_count(W_V2_60, *TEST, min_profit=0.20, vol_mult=2,
                                       capital_usd=CAP_USD, min_order_usd=MIN_ORDER)
print(f"+trailing-stop: order_dilewati (< $5) = {r['n_skipped']} dari {r['n_decisions']} keputusan "
      f"({r['n_stop']} trigger stop, {r['n_reb']} event rebalance)")

Baseline V2-60: order_dilewati (< $5) = 225
+trailing-stop: order_dilewati (< $5) = 183 dari 455 keputusan (32 trigger stop, 74 event rebalance)


**Dugaan pertama ini SALAH** — order yang dilewati karena < $5 justru **lebih sedikit** di versi
trailing-stop (183) dibanding baseline (225), bukan lebih banyak. Bukan itu penyebabnya. Lanjut cek
kemungkinan lain: apakah hasil "baseline menang" ini memang robust, atau cuma kebetulan di titik
modal/ambang order yang persis itu.

### 5b. Cek sensitivitas: apa hasilnya berubah drastis cuma krn modal/ambang order beda tipis?

In [11]:
def quick_stats(eq):
    dr = eq.pct_change().dropna()
    return dict(sharpe=dr.mean() / dr.std() * np.sqrt(365), max_dd=(eq / eq.cummax() - 1).min())

rows_sens = []
for modal, min_order in [(1_500_000, 5.0), (1_400_000, 5.0), (1_600_000, 5.0),
                         (1_500_000, 4.0), (1_500_000, 6.0), (3_000_000, 5.0), (10_000_000, 5.0)]:
    cap = modal / KURS
    eqb, _ = run_portfolio(P, W_V2_60, *TEST, capital_usd=cap, min_order_usd=min_order)
    eqt, _ = run_portfolio_trailing_stop(P, W_V2_60, *TEST, min_profit=0.20, vol_mult=2,
                                          capital_usd=cap, min_order_usd=min_order)
    sb, st = quick_stats(eqb), quick_stats(eqt)
    rows_sens.append(dict(modal=f"Rp{modal:,.0f}", min_order=f"${min_order:.0f}",
                           sharpe_baseline=sb["sharpe"], sharpe_trailing=st["sharpe"],
                           dd_baseline=sb["max_dd"], dd_trailing=st["max_dd"],
                           pemenang="trailing" if st["sharpe"] > sb["sharpe"] else "baseline"))
sens = pd.DataFrame(rows_sens).set_index(["modal", "min_order"])
out = sens.copy().astype(object)
for c in ("sharpe_baseline", "sharpe_trailing"):
    out[c] = sens[c].map(lambda v: f"{v:.2f}")
for c in ("dd_baseline", "dd_trailing"):
    out[c] = sens[c].map(lambda v: f"{v:+.0%}")
out

sharpe_baseline sharpe_trailing dd_baseline dd_trailing  pemenang
modal        min_order                                                                  
Rp1,500,000  $5                   1.16            1.14        -19%        -21%  baseline
Rp1,400,000  $5                   1.09            1.13        -21%        -21%  trailing
Rp1,600,000  $5                   1.14            1.25        -20%        -18%  trailing
Rp1,500,000  $4                   1.16            1.28        -22%        -19%  trailing
             $6                   1.09            1.05        -21%        -18%  baseline
Rp3,000,000  $5                   1.15            1.26        -22%        -19%  trailing
Rp10,000,000 $5                   1.17            1.28        -21%        -19%  trailing

**Ketemu penjelasan yang benar**: hasilnya **sangat sensitif** terhadap perubahan kecil pada modal
atau ambang order minimum — Rp1,4 juta dan Rp1,6 juta (tetangga dekat Rp1,5 juta) sama-sama membuat
trailing-stop **menang lagi**; begitu juga $4 (menang) vs $6 (kalah) di ambang order yang cuma beda
$1; dan di modal lebih besar (Rp3 juta, Rp10 juta) trailing-stop konsisten menang. Titik
"Rp1,5 juta + $5" yang kebetulan dipakai notebook 19/20 sebagai acuan ternyata persis jatuh di salah
satu titik yang kebetulan membalik — bukan tanda sinyal menghilang secara sistematis di modal kecil.

## Kesimpulan

**Revisi dari kesimpulan awal**: bukan "trailing-stop terbukti membantu" (klaim pertama, dari hasil
skala modal besar saja) dan bukan juga "trailing-stop ditolak di modal Rp1,5 juta" (klaim kedua, dari
satu titik modal/order saja). Setelah diselidiki lebih jauh (atas pertanyaan user): **hasil di periode
uji terlalu sensitif terhadap detail kecil (rounding order minimum $5, pembulatan modal) untuk
disimpulkan dengan percaya diri ke arah manapun** — baik "membantu" maupun "merugikan".

### Apa yang sudah pasti
1. **Di periode dev (2018-07→2022-12, sampel lebih besar)**, trailing-stop (profit≥20%, 2×ATR-14)
   **konsisten unggul** di hampir semua skenario modal yang dicoba (Sharpe dev 1.68→1.77 dengan
   batasan modal nyata, serupa dengan tanpa batasan).
2. **Di periode uji (2023-01→2026-06)**, jumlah kejadian trigger stop cuma ±30-40× dalam 3.5 tahun —
   sampel ini kecil, dan hasil Sharpe-nya **berganti pemenang** hanya dengan menggeser modal ±Rp100rb
   atau ambang order ±$1 (lihat tabel sensitivitas di atas). Ini pertanda **sinyal yang diukur di sini
   lebih kecil dari noise eksekusi** pada skala sampel sekecil ini — bukan efek "modal kecil = gagal"
   yang sistematis (modal Rp1,4jt & Rp1,6jt, sama-sama "kecil", malah menang).
3. Dugaan pertama penulis (order kecil lebih sering di-skip di versi trailing-stop) **terbukti salah**
   saat dicek langsung (183 skip vs 225 baseline, trailing-stop justru lebih sedikit skip).

### Catatan jujur / batasan
1. Jumlah kejadian stop di periode uji kecil (±30-40, 8 coin, 3.5 tahun) — terlalu sedikit buat
   mengukur efek net yang presisi melalui backtest historis saja.
2. Hanya ATR(14) yang dicoba sebagai ukuran volatilitas; belum dicoba lookback/ukuran lain.
3. **Simulasi backtest, bukan jaminan hasil nyata** — berlaku dobel di sini karena sinyalnya sendiri
   terbukti sensitif terhadap detail kecil.

### Rekomendasi
**Belum bisa disimpulkan dari backtest historis saja apakah trailing-stop per-coin menambah atau
mengurangi hasil di modal Rp1,5 juta** — beda dengan filter per-coin (notebook 21) yang memang
konsisten gagal di banyak variasi/skala. Trailing-stop ini **masuk kategori "belum terbukti ke arah
manapun"**, bukan "terbukti gagal". Kalau mau dipakai, perlakukan sebagai eksperimen yang perlu
divalidasi lewat **paper trading / testnet** (bukan cuma backtest tambahan) sebelum dipakai modal
riil — konsisten dengan prinsip README bagian 7 poin 1. Jangan jadikan backtest historis saja sebagai
dasar keputusan akhir untuk fitur ini.